# Supplementary Figure — same-data versus participant-held-out sparse selection

This figure directly visualizes the circularity control.

The same-data analysis ranks and evaluates archetypes using the same participant
set. The held-out analysis ranks on one participant half and evaluates the
selected subset on the other half, then reverses the halves.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SAME_CSV = Path(
    "msaa_condrank_decoding_outputs_spatial_across/topm_summary.csv"
)
HELDOUT_CSV = Path(
    "msaa_heldout_topm_outputs_spatial_across/heldout_topm_crossfit_runs.csv"
)

CONDITIONS = ["intact", "word", "rest"]
COLORS = {
    "intact": "#7B3294",
    "word": "#2CA25F",
    "rest": "#222222",
}
TOP_M_VALUES = [5, 10]

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_same_data_vs_heldout_topm.png"
FIG_PDF = OUTPUT_DIR / "Supplement_same_data_vs_heldout_topm.pdf"

for p in [SAME_CSV, HELDOUT_CSV]:
    if not p.exists():
        raise FileNotFoundError(p)

same = pd.read_csv(SAME_CSV)
held = pd.read_csv(HELDOUT_CSV)

In [ ]:
def sem(x):
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))

held_outer = (
    held
    .groupby(
        ["condition", "K", "top_m", "outer_rep"],
        as_index=False,
    )
    .agg(
        mean=("mean", "mean"),
    )
)

held_summary = (
    held_outer
    .groupby(
        ["condition", "K", "top_m"],
        as_index=False,
    )
    .agg(
        mean=("mean", "mean"),
        err=("mean", sem),
    )
)

same = same.copy()
if "sem" in same.columns and "err" not in same.columns:
    same = same.rename(columns={"sem": "err"})

display(held_summary.head())

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.2, 4.9),
    sharey=False,
)

for ax, top_m, letter in zip(
    axes,
    TOP_M_VALUES,
    ["A", "B"],
):
    for condition in CONDITIONS:
        color = COLORS[condition]

        a = (
            same[
                (same["condition"].astype(str) == condition)
                & (same["top_m"].astype(int) == top_m)
            ]
            .sort_values("K")
        )

        b = (
            held_summary[
                (held_summary["condition"].astype(str) == condition)
                & (held_summary["top_m"].astype(int) == top_m)
            ]
            .sort_values("K")
        )

        if len(a):
            ax.plot(
                a["K"],
                a["mean"],
                color=color,
                linestyle=":",
                linewidth=1.8,
                alpha=0.70,
            )

        if len(b):
            ax.errorbar(
                b["K"],
                b["mean"],
                yerr=b["err"],
                color=color,
                linestyle="-",
                marker="o",
                markersize=3.5,
                linewidth=2.0,
                capsize=2,
            )

    ax.set_title(
        f"top-{top_m}",
        fontweight="bold",
        fontsize=13,
    )
    ax.set_xlabel("Model order K")
    ax.set_ylabel("Timepoint-identification accuracy")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.text(
        -0.10, 1.04, letter,
        transform=ax.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
    )

condition_handles = [
    plt.Line2D([0], [0], color=COLORS[c], lw=2, label=c.capitalize())
    for c in CONDITIONS
]

design_handles = [
    plt.Line2D([0], [0], color="0.25", lw=1.8, linestyle=":",
               label="Same-data ranking/evaluation"),
    plt.Line2D([0], [0], color="0.25", lw=2.0, linestyle="-", marker="o",
               label="Participant-held-out ranking/evaluation"),
]

fig.legend(
    handles=condition_handles + design_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    "Sparse-archetype performance before and after participant-level cross-fitting",
    fontsize=15,
    fontweight="bold",
)

fig.subplots_adjust(bottom=0.20, top=0.86, wspace=0.25)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)